# House Price Analysis: Cleaning, Visualization & Hypothesis Testing
Mini Project Notebook (Google Colab) — structure follows the *A/B Test Pricing Report* reference.

**Sections:** 1 Introduction · 2 Data Description & Cleaning · 3 Methodology · 4 Results (EDA + tests) · 5 Discussion · 6 Conclusion

**How to run:** Runtime → Run all. When prompted, upload `House_Price_data.csv`.

## 1. Introduction
The business question: **which property characteristics are statistically associated with house price, and how large are the effects?** As in the reference report, three complementary tests are cross-checked (t-test, chi-square, ANOVA), and a regression is added to control for confounding.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.options.display.float_format = "{:,.2f}".format

try:
    from google.colab import files
    up = files.upload()                       # choose House_Price_data.csv
    PATH = list(up.keys())[0]
except ImportError:
    PATH = "House_Price_data.csv"             # running outside Colab

raw = pd.read_csv(PATH, parse_dates=["date"])
print(raw.shape)
raw.head()

## 2. Data Description

In [ ]:
raw.info()
display(raw.describe().T)
print("Missing values:", raw.isnull().sum().sum())
print("Exact duplicate rows:", raw.duplicated().sum())
print("Sale dates:", raw.date.min().date(), "to", raw.date.max().date())
print("Cities:", raw.city.nunique(), "| Zip codes:", raw.statezip.nunique(), "| Countries:", raw.country.unique())

### 2.1 Data Cleaning
There are no missing values, but `describe()` exposes real quality problems. Each rule is logged so the effect of every step is visible (like the report's group/page-mismatch and duplicate-user steps).

In [ ]:
df = raw.copy()
log = []
def drop_rows(name, mask):
    global df
    n = int(mask.sum()); df = df[~mask]
    log.append({"Cleaning step": name, "Rows removed": n, "Rows remaining": len(df)})

# 1. Price of $0 is a placeholder, not a real sale
drop_rows("price == 0", df.price == 0)
# 2. 0 bedrooms or 0 bathrooms on a house is a data-entry error
drop_rows("bedrooms == 0 or bathrooms == 0", (df.bedrooms == 0) | (df.bathrooms == 0))
# 3. Impossible price per sq ft (e.g. $26.6M for 1,180 sqft; $7,800 for 780 sqft) -> 3xIQR on log scale
df = df.assign(pps = df.price / df.sqft_living)
lp = np.log(df.pps); q1, q3 = lp.quantile([.25, .75]); iqr = q3 - q1
drop_rows("price per sqft outlier (3xIQR, log)", (lp < q1 - 3*iqr) | (lp > q3 + 3*iqr))
# 4. Extreme prices on log scale
lp = np.log(df.price); q1, q3 = lp.quantile([.25, .75]); iqr = q3 - q1
drop_rows("price outlier (3xIQR, log)", (lp < q1 - 3*iqr) | (lp > q3 + 3*iqr))
# 5. Same street + city + date listed twice (repeated streets on different dates are different units, so kept)
drop_rows("duplicate listing (street+city+date)", df.duplicated(["street", "city", "date"]))

display(pd.DataFrame(log))
print(f"Rows: {len(raw):,} -> {len(df):,}  ({len(raw)-len(df)} removed, {(len(raw)-len(df))/len(raw):.1%})")

In [ ]:
# Feature engineering + tidy up
df = df.drop(columns="country").copy()               # constant column (all 'USA')
df["log_price"]    = np.log(df.price)                # price is right-skewed
df["renovated"]    = (df.yr_renovated > 0).astype(int)   # yr_renovated == 0 means never renovated
df["has_basement"] = (df.sqft_basement > 0).astype(int)
df["has_view"]     = (df.view > 0).astype(int)
df["house_age"]    = 2014 - df.yr_built              # data are from 2014 sales
df["high_price"]   = (df.price > df.price.median()).astype(int)
df.describe().T[["count","mean","std","min","50%","max"]]

## 3. Methodology
All tests use α = 0.05. H0 = no effect / no relationship; H1 = there is an effect.

| Test | Question | Variables |
|---|---|---|
| Welch t-test | Do renovated and non-renovated houses differ in price? | `renovated` × `log_price` |
| Chi-square | Is having a view related to being in the top-half of prices? | `has_view` × `high_price` |
| One-way ANOVA | Does price differ by condition rating and by city? | `condition`, `city` × `log_price` |
| OLS regression | Which effects survive after controlling for size, age, etc.? | `log_price` ~ features |

Price is right-skewed, so tests use **log(price)**; non-parametric checks (Mann-Whitney, Kruskal-Wallis) confirm robustness.

## 4. Results
### 4.1 Descriptive Overview & Visualization

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(raw.price[raw.price < 3e6], bins=60, ax=ax[0], color="#c0392b")
ax[0].set_title("BEFORE cleaning (spike at $0, prices < $3M shown)")
sns.histplot(df.price, bins=60, kde=True, ax=ax[1])
ax[1].set_title("AFTER cleaning")
for a in ax: a.set_xlabel("Price ($)")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df.price, bins=50, kde=True, ax=ax[0]); ax[0].set_title(f"Price (skew = {df.price.skew():.2f})")
sns.histplot(df.log_price, bins=50, kde=True, ax=ax[1], color="green"); ax[1].set_title(f"log(Price) (skew = {df.log_price.skew():.2f})")
plt.tight_layout(); plt.show()

In [ ]:
num = ["price","bedrooms","bathrooms","sqft_living","sqft_lot","floors","waterfront","view","condition","sqft_basement","house_age"]
plt.figure(figsize=(9, 7))
sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix"); plt.show()
print(df[num].corr()["price"].drop("price").sort_values(ascending=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
top10 = df.groupby("city").price.agg(["median","count"]).query("count >= 30").sort_values("median", ascending=False).head(10)
top10["median"].sort_values().plot(kind="barh", ax=ax[0], color="#2980b9"); ax[0].set_title("Median price: top 10 cities (n >= 30)")
sns.scatterplot(data=df, x="sqft_living", y="price", hue="has_view", alpha=.5, ax=ax[1]); ax[1].set_title("Living area vs price")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=df, x="bedrooms", y="price", ax=ax[0]); ax[0].set_ylim(0, 2.5e6); ax[0].set_title("Price by bedrooms")
sns.boxplot(data=df, x="bathrooms", y="price", ax=ax[1]); ax[1].set_ylim(0, 2.5e6); ax[1].set_title("Price by bathrooms"); ax[1].tick_params(axis="x", rotation=90)
sns.boxplot(data=df, x="floors", y="price", ax=ax[2]); ax[2].set_ylim(0, 2.5e6); ax[2].set_title("Price by floors")
plt.tight_layout(); plt.show()

### 4.2 T-Test: Renovated vs Not Renovated

In [ ]:
a = df.loc[df.renovated == 1, "log_price"]; b = df.loc[df.renovated == 0, "log_price"]
t, p_t = stats.ttest_ind(a, b, equal_var=False)
diff = a.mean() - b.mean(); se = np.sqrt(a.var()/len(a) + b.var()/len(b))
d = diff / np.sqrt((a.var() + b.var())/2)
mw = stats.mannwhitneyu(a, b).pvalue
res = pd.DataFrame({"Metric": ["n renovated / not", "Median price renovated", "Median price not renovated",
        "Difference (log)  -> % difference", "95% CI of difference (log)", "t-statistic", "p-value (Welch)",
        "Cohen's d", "p-value (Mann-Whitney, robustness)"],
    "Value": [f"{len(a):,} / {len(b):,}", f"${df[df.renovated==1].price.median():,.0f}", f"${df[df.renovated==0].price.median():,.0f}",
        f"{diff:.3f}  ->  {np.exp(diff)-1:.1%}", f"[{diff-1.96*se:.3f}, {diff+1.96*se:.3f}]", f"{t:.3f}", f"{p_t:.2e}", f"{d:.3f}", f"{mw:.2e}"]})
display(res)
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x="renovated", y="price", ax=ax, showfliers=False); ax.set_xticks([0, 1]); ax.set_xticklabels(["Not renovated", "Renovated"])
ax.set_title("Price by renovation status"); plt.show()
print(df.groupby("renovated")[["sqft_living","house_age"]].median().rename(index={0:"Not renovated",1:"Renovated"}))

### 4.3 Chi-Square: View vs High-Price Home

In [ ]:
ct = pd.crosstab(df.has_view, df.high_price)
ct.index = ["No view", "Has view"]; ct.columns = ["Below median", "Above median"]
display(ct)
chi2, p_chi, dof, exp = stats.chi2_contingency(ct)
v = np.sqrt(chi2 / ct.values.sum())
print(f"chi2 = {chi2:.2f} | dof = {dof} | p = {p_chi:.2e} | Cramer's V = {v:.3f}")
share = df.groupby("has_view").high_price.mean().rename({0:"No view",1:"Has view"})
share.plot(kind="bar", color=["#95a5a6","#27ae60"], rot=0); plt.ylabel("Share above median price"); plt.title("Homes with a view are far more often in the upper price half"); plt.show()
print(share.map("{:.1%}".format))

### 4.4 ANOVA: Price by Condition and by City

In [ ]:
# (a) by condition rating
groups = [g.log_price.values for _, g in df.groupby("condition")]
F1, p1 = stats.f_oneway(*groups); H1, pk1 = stats.kruskal(*groups)
display(df.groupby("condition").price.agg(count="count", median="median", mean="mean"))
# (b) by city (top 5 cities by number of sales)
top5 = df.city.value_counts().head(5).index; s = df[df.city.isin(top5)]
groups = [g.log_price.values for _, g in s.groupby("city")]
F2, p2 = stats.f_oneway(*groups); H2, pk2 = stats.kruskal(*groups)
eta2 = sum(len(g)*(g.mean()-s.log_price.mean())**2 for g in groups) / ((s.log_price-s.log_price.mean())**2).sum()
display(s.groupby("city").agg(sales=("price","count"), median_price=("price","median"), median_pps=("pps","median")))
display(pd.DataFrame({"Comparison": ["Condition (5 levels)", "City (top 5)"], "F-statistic": [F1, F2], "ANOVA p": [p1, p2], "Kruskal p": [pk1, pk2]}))
print(f"Eta-squared (city): {eta2:.3f}  -> city explains ~{eta2:.0%} of the variance in log price")
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=df, x="condition", y="price", ax=ax[0], showfliers=False); ax[0].set_title("Price by condition")
sns.boxplot(data=s, x="city", y="price", ax=ax[1], showfliers=False, order=s.groupby("city").price.median().sort_values().index); ax[1].set_title("Price by city (top 5)")
plt.tight_layout(); plt.show()

### 4.5 Regression: Which Effects Survive Controls?

In [ ]:
cols = ["sqft_living","bedrooms","bathrooms","floors","waterfront","view","condition","house_age","renovated"]
X = np.column_stack([np.ones(len(df)), df[cols].values]); y = df.log_price.values
beta = np.linalg.lstsq(X, y, rcond=None)[0]; resid = y - X @ beta; n, k = X.shape
s2 = resid @ resid / (n - k); se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
tvals = beta / se; pvals = 2 * stats.t.sf(np.abs(tvals), n - k)
r2 = 1 - resid @ resid / ((y - y.mean()) @ (y - y.mean()))
out = pd.DataFrame({"coef": beta, "t": tvals, "p": pvals, "approx % effect on price": (np.exp(beta) - 1) * 100}, index=["intercept"] + cols)
display(out.drop(index="intercept"))
print(f"R-squared = {r2:.3f}")

## 5. Discussion
See the interpretation block below; it is computed from the results above so the numbers always match the cleaned data.

In [ ]:
print("INTERPRETATION")
print("-"*70)
print(f"1. Cleaning removed {len(raw)-len(df)} of {len(raw):,} rows ({(len(raw)-len(df))/len(raw):.1%}); the largest problem was {int((raw.price==0).sum())} sales recorded at $0.")
print(f"2. Price is right-skewed (skew {df.price.skew():.2f}); log transform brings it to {df.log_price.skew():.2f}, so tests use log(price).")
print(f"3. Size dominates: sqft_living has correlation {df[['price','sqft_living']].corr().iloc[0,1]:.2f} with price; bathrooms {df[['price','bathrooms']].corr().iloc[0,1]:.2f}.")
print(f"4. Renovation (raw): p = {p_t:.1e} -> renovated homes look {abs(np.exp(diff)-1):.0%} CHEAPER. This is confounded: they are older ({df[df.renovated==1].house_age.median():.0f} vs {df[df.renovated==0].house_age.median():.0f} yrs) and smaller.")
print(f"   After regression controls, renovation p = {out.loc['renovated','p']:.2f} -> no significant independent effect.")
print(f"5. View x high price: chi2 p = {p_chi:.1e}, Cramer's V = {v:.2f} (moderate association).")
print(f"6. City matters a lot: ANOVA p = {p2:.1e}, eta2 = {eta2:.2f}.")
print(f"7. The regression explains R2 = {r2:.2f} of log-price variance; remaining variance is mostly exact location/neighbourhood, which these columns capture only via city.")

## 6. Conclusion & Recommendations
* Clean first: $0 prices and impossible price-per-sqft values must be removed or every average is distorted.
* Living area, bathrooms, view, waterfront and city are the strongest and most statistically reliable price drivers.
* Do **not** conclude that renovation lowers price. The raw gap is a confounding effect of age and size; after controls it is not significant.
* **Limitations:** only ~10 weeks of sales (May–July 2014), one metro area, and statistical significance ≠ causation. Very large samples make small effects "significant", so always read effect sizes (Cohen's d, Cramér's V, η²).
* **Next steps:** add zip-code effects, try Random Forest / gradient boosting for prediction, and validate on a hold-out set.

In [ ]:
# Save the cleaned dataset
df.to_csv("House_Price_cleaned.csv", index=False)
try:
    files.download("House_Price_cleaned.csv")
except NameError:
    pass